In [ ]:
# Import repo files
!git clone -b hao https://github.com/sahitidoke/asym-model-simulation.git

Cloning into 'asym-model-simulation'...
remote: Enumerating objects: 551, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 551 (delta 149), reused 182 (delta 87), pack-reused 300 (from 1)
Receiving objects: 100% (551/551), 3.15 MiB | 8.22 MiB/s, done.
Resolving deltas: 100% (300/300), done.


In [ ]:
!cd /content/asym-model-simulation && git branch --show-current

hao


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [33]:
import os
import sys
sys.path.append('/content/asym-model-simulation')
import contextlib
import numpy as np
import json
from sklearn.covariance import graphical_lasso
from matplotlib import pyplot as plt

from method import EM_algorithm as em, tlasso
from simulation import simulation_data_generator as dg

COLOR_ASYM = "#2a78d6"
COLOR_EM_DIAG = "#f5239a"
COLOR_GGM = "#e34948"
COLOR_T = "#f5a623"
COLOR_TS = "#23f57e"
COLOR_CHANCE = "#c3c2b7"


def offdiag_max(S):
    """max_{i != j} |S_ij|. The smallest rho that zeroes every off-diagonal."""
    A = np.abs(np.asarray(S, dtype=float)).copy()
    np.fill_diagonal(A, 0.0)
    rmax = float(A.max())
    if not np.isfinite(rmax) or rmax <= 0:
        raise ValueError("rho_max is not positive; check the input matrix.")
    return rmax


def make_rho_grid(rho_max, n_rho=50, max_ratio = 1, min_ratio=0.05):
    """Log-spaced, DESCENDING grid on [min_ratio * rho_max, rho_max].

    Log spacing because edge count is roughly geometric in rho: linear
    spacing wastes most points in the dense end where the ROC curve barely
    moves. Descending so warm starts run sparse -> dense, which is the
    numerically stable direction when p > n.
    """
    return np.logspace(np.log10(max_ratio * rho_max), np.log10(min_ratio * rho_max), n_rho)


def log_step_ratio(rho_grid):
    """Multiplicative step of a log-spaced grid, so an extension stays even.

    A descending grid gives ratio < 1; multiplying the last rho by it
    repeatedly continues the same geometric sequence past the bottom of the
    grid, which is what the fp_max extension in roc_curve_em walks along.
    """
    g = np.asarray(rho_grid, dtype=float)
    if g.size < 2:
        raise ValueError("need at least two rho values to infer the step")
    ratio = float((g[-1] / g[0]) ** (1.0 / (g.size - 1)))
    if not 0.0 < ratio < 1.0:
        raise ValueError("rho grid must be strictly descending to be extended")
    return ratio


def pilot_rho_max(
    Y, algorithm, rho, algorithm_kwargs=None, rho_name="rho", S_key="S_tau",
):
    """rho_max for one method on one replicate: fit at `rho`, read off its S.

    The EM M-step penalizes the working covariance S_tau, not cov(Y): S_tau is
    tau-weighted, skew-corrected and (for MWGP) Monte-Carlo averaged, so its
    off-diagonal scale is method-specific. Building every method's grid from
    cov(Y) therefore starts each path at a different point along its own
    regularization range -- some methods begin already empty, others begin
    dense, and the index-wise average across replicates mixes those.

    S_tau also depends on rho, so there is no rho-free version of it to
    calibrate against. The pilot fit at the theoretical rho = sqrt(log p / n)
    is the reference point: run the EM there to convergence, take
    max_{i != j} |S_ij| of the converged S_tau, and use that as rho_max.
    """
    kwargs = {} if algorithm_kwargs is None else dict(algorithm_kwargs)
    print(f"  pilot fit of {algorithm.__name__} at rho={rho:.5f}")
    res = algorithm(Y, **{rho_name: rho}, **kwargs)
    rho_max = offdiag_max(res[S_key])
    print(f"  -> rho_max = max|S_ij| (off-diag) = {rho_max:.5f}")
    return rho_max


def edge_confusion(Theta_hat, true_pos_mask, true_neg_mask, tol=1e-8):
    iu = np.triu_indices(Theta_hat.shape[0], k=1)
    est_edges = np.abs(Theta_hat[iu]) > tol
    tp_rate = np.sum(est_edges & true_pos_mask) / true_pos_mask.sum()
    fp_rate = np.sum(est_edges & true_neg_mask) / true_neg_mask.sum()
    return fp_rate, tp_rate


@contextlib.contextmanager
def stream_to(path, also_stderr=True):
    """Send print()/warnings to `path` instead of the cell output.

    buffering=1 is the point: line buffering means the file fills as the run
    goes, so `Get-Content <path> -Wait` follows it live. On an exception the
    streams are restored before the traceback renders, so errors still show
    up in the notebook.
    """
    f = open(path, "w", buffering=1, encoding="utf-8")
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = f
    if also_stderr:
        sys.stderr = f
    try:
        yield f
    finally:
        sys.stdout, sys.stderr = old_out, old_err
        f.close()


def roc_curve_em(
    Y, rho_grid, algorithm, true_pos_mask, true_neg_mask,
    algorithm_kwargs=None, rho_name="rho", theta_key="Theta",
    fp_max=None, max_extra=50,
):
    """Re-run the whole EM at each rho, warm-started, sparsest first.

    `algorithm_kwargs` holds whatever extra arguments the given algorithm takes
    (e.g. nu/n_iter for run_tlasso, n_burn/n_keep for run_em_MWGP); `rho_name`
    and `theta_key` cover algorithms that name the penalty or the returned
    precision matrix differently.

    Each fit is initialized at the previous rho's mu and Theta. Only those two
    are carried: nu/eta are tail-shape nuisance parameters, and handing them
    forward lets nu ratchet down across the sweep until lam = -2/nu - 0.5 is
    negative enough to overflow the Bessel terms in gig_moment.

    These EM objectives are not convex, so warm starting changes the estimator
    and not just the runtime: every point on the curve depends on the whole
    path prefix, and the sweep must stay sparse -> dense for the results to be
    reproducible.

    `fp_max` makes the dense end of the path data-driven rather than fixed. A
    grid that stops at min_ratio * rho_max stops wherever that happens to land
    in FPR, which differs by method and by replicate; the curve is then
    truncated and mean_roc has to interpolate straight to (1, 1) across
    everything above it. With fp_max set, the sweep keeps stepping past the end
    of rho_grid -- same multiplicative step, so the log spacing is unbroken --
    until the false positive rate reaches fp_max or `max_extra` extra fits have
    run. Each extra point is warm started from the previous one exactly like
    the base grid, so the path stays a single sparse -> dense sweep.

    Returns (fp, tp, rho_used). rho_used is rho_grid plus whatever extension
    was needed, so it -- not rho_grid -- is the grid the caller must keep.
    """
    kwargs = {} if algorithm_kwargs is None else dict(algorithm_kwargs)
    rhos = [float(r) for r in rho_grid]
    n_base = len(rhos)
    ratio = log_step_ratio(rhos) if fp_max is not None else None
    fp, tp = [], []
    prev = None
    tail = f", extending until FPR >= {fp_max}" if fp_max is not None else ""
    print(f"Running {algorithm.__name__} over {n_base} rho values{tail}...")
    i = 0
    while i < len(rhos):
        extra = "" if i < n_base else f"  [extra {i - n_base + 1}/{max_extra}]"
        print(f"  rho={rhos[i]:.5f} ({i+1}/{len(rhos)}){extra}")

        res = algorithm(
            Y, **{rho_name: rhos[i]}, **kwargs,
            **({"init": prev} if prev is not None else {}),
        )
        Theta_hat = res[theta_key]
        if np.all(np.isfinite(Theta_hat)):
            prev = {"mu": res["mu"], "Theta": Theta_hat}

        fp_i, tp_i = edge_confusion(Theta_hat, true_pos_mask, true_neg_mask)
        fp.append(fp_i)
        tp.append(tp_i)
        print(fp_i, tp_i)
        i += 1

        # Only ever extend off the current last point, so the added fits stay
        # in the same warm-started sparse -> dense order as the base grid.
        if (fp_max is not None and i == len(rhos) and fp_i < fp_max
                and i - n_base < max_extra):
            rhos.append(rhos[-1] * ratio)

    if fp_max is not None and fp[-1] < fp_max:
        print(f"  [warn] FPR stalled at {fp[-1]:.4f} < fp_max={fp_max} after "
              f"{len(rhos) - n_base} extra fits (cap max_extra={max_extra}); "
              f"the curve stops short of fp_max")
    return np.asarray(fp), np.asarray(tp), np.asarray(rhos)


def roc_curve_em_autogrid(
    Y, algorithm, true_pos_mask, true_neg_mask, pilot_rho, n_rho,
    min_ratio=0.05, max_ratio=1, algorithm_kwargs=None, rho_name="rho",
    theta_key="Theta", S_key="S_tau", fp_max=None, max_extra=50,
):
    """pilot_rho_max -> make_rho_grid -> roc_curve_em, for one method/replicate.

    Returns (fp, tp, rho_grid); the grid is method- and replicate-specific and
    has to be kept, since the theoretical rho no longer sits at a common index.
    With fp_max set it is also variable-length -- n_rho is the starting size,
    and roc_curve_em appends points below min_ratio * rho_max until the FPR
    reaches fp_max -- so the returned grid is the one that was actually swept.

    The pilot fit is NOT reused as a warm start: the sweep must run
    sparse -> dense from its own top end for the path to be reproducible, and
    the pilot sits in the middle of it.
    """
    rho_max = pilot_rho_max(
        Y, algorithm, pilot_rho, algorithm_kwargs=algorithm_kwargs,
        rho_name=rho_name, S_key=S_key,
    )
    if pilot_rho > rho_max:
        print(f"  [warn] pilot rho {pilot_rho:.5f} > rho_max {rho_max:.5f}: "
              f"it falls off the top of this method's grid")
    rho_grid = make_rho_grid(
        rho_max, n_rho=n_rho, max_ratio=max_ratio, min_ratio=min_ratio,
    )
    print(f"  rho grid: {rho_grid}")
    fp, tp, rho_used = roc_curve_em(
        Y, rho_grid, algorithm, true_pos_mask, true_neg_mask,
        algorithm_kwargs=algorithm_kwargs, rho_name=rho_name,
        theta_key=theta_key, fp_max=fp_max, max_extra=max_extra,
    )
    if len(rho_used) > len(rho_grid):
        print(f"  grid extended by {len(rho_used) - len(rho_grid)} points "
              f"down to rho={rho_used[-1]:.5f} to reach FPR >= {fp_max}")
    return fp, tp, rho_used


def roc_curve_glasso(Y, rho_grid, true_pos_mask, true_neg_mask,
                     glasso_kwargs=None, fp_max=None, max_extra=50):
    """roc_curve_full_em analogue for the naive Gaussian glasso baseline.

    sklearn's graphical_lasso takes an empirical covariance rather than Y,
    names the penalty `alpha`, and returns (covariance, precision), so it needs
    its own loop. `fp_max` / `max_extra` extend the grid exactly as in
    roc_curve_em, and (fp, tp, rho_used) is returned for the same reason.
    """
    kwargs = {} if glasso_kwargs is None else dict(glasso_kwargs)
    S = np.cov(Y, rowvar=False) + 1e-10 * np.eye(Y.shape[1])
    p = Y.shape[1]
    rhos = [float(r) for r in rho_grid]
    n_base = len(rhos)
    ratio = log_step_ratio(rhos) if fp_max is not None else None
    fp, tp = [], []
    Theta_prev = np.eye(p)
    tail = f", extending until FPR >= {fp_max}" if fp_max is not None else ""
    print(f"Running graphical_lasso over {n_base} rho values{tail}...")
    i = 0
    while i < len(rhos):
        extra = "" if i < n_base else f"  [extra {i - n_base + 1}/{max_extra}]"
        print(f"  rho={rhos[i]:.5f} ({i+1}/{len(rhos)}){extra}")
        try:
            _, Theta_hat = graphical_lasso(S + rhos[i] * np.eye(p), alpha=rhos[i], **kwargs)
        except Exception:
            Theta_hat = Theta_prev
        Theta_prev = Theta_hat
        fp_i, tp_i = edge_confusion(Theta_hat, true_pos_mask, true_neg_mask)
        fp.append(fp_i)
        tp.append(tp_i)
        i += 1

        if (fp_max is not None and i == len(rhos) and fp_i < fp_max
                and i - n_base < max_extra):
            rhos.append(rhos[-1] * ratio)

    if fp_max is not None and fp[-1] < fp_max:
        print(f"  [warn] FPR stalled at {fp[-1]:.4f} < fp_max={fp_max} after "
              f"{len(rhos) - n_base} extra fits (cap max_extra={max_extra}); "
              f"the curve stops short of fp_max")
    return np.asarray(fp), np.asarray(tp), np.asarray(rhos)


def roc_curve_glasso_autogrid(
    Y, true_pos_mask, true_neg_mask, n_rho, min_ratio=0.05, max_ratio=1,
    glasso_kwargs=None, fp_max=None, max_extra=50,
):
    """roc_curve_em_autogrid for the Gaussian baseline; no pilot fit needed.

    glasso penalizes the empirical covariance itself, which does not depend on
    rho, so its rho_max is available in closed form -- the pilot fit would
    return the same S it was handed.
    """
    S = np.cov(Y, rowvar=False)
    rho_max = offdiag_max(S)
    print(f"  graphical_lasso rho_max = max|S_ij| (off-diag) = {rho_max:.5f}")
    rho_grid = make_rho_grid(
        rho_max, n_rho=n_rho, max_ratio=max_ratio, min_ratio=min_ratio,
    )
    print(f"  rho grid: {rho_grid}")
    fp, tp, rho_used = roc_curve_glasso(
        Y, rho_grid, true_pos_mask, true_neg_mask, glasso_kwargs=glasso_kwargs,
        fp_max=fp_max, max_extra=max_extra,
    )
    if len(rho_used) > len(rho_grid):
        print(f"  grid extended by {len(rho_used) - len(rho_grid)} points "
              f"down to rho={rho_used[-1]:.5f} to reach FPR >= {fp_max}")
    return fp, tp, rho_used


def auc_from_curve(fp, tp):
    order = np.argsort(fp)
    return np.trapezoid(tp[order], fp[order])


def mean_roc(fp, tp, n_points=201):
    """Vertical average of the replicate ROC curves (Fawcett 2006, Sec. 6.1).

    Averaging fp and tp down a column would be threshold averaging, which needs
    column i to be the same threshold in every row. It is not: every replicate
    has its own rho grid built from its own rho_max, so column i is a different
    rho in every row. Averaging it smears the curve horizontally, and where the
    curves are convex the mean point can land below all of them.

    Interpolating each replicate's TPR onto a common FPR grid and averaging
    vertically needs no correspondence between the rho grids at all. Returns
    (fp_common, tp_mean, tp_se).

    `fp` and `tp` are sequences of per-replicate curves, not necessarily a
    rectangular array: with fp_max set, each replicate's grid stops where its
    own FPR reached fp_max, so the curves differ in length.
    """
    fp = [np.asarray(f, dtype=float) for f in fp]
    tp = [np.asarray(t, dtype=float) for t in tp]
    n_rep = len(fp)
    fp_common = np.linspace(0.0, 1.0, n_points)
    tp_interp = np.empty((n_rep, n_points))
    for r in range(n_rep):
        # Anchor at (0,0) and (1,1): the path stops at min_ratio * rho_max, so
        # it need not reach either corner on its own, and np.interp would
        # otherwise extrapolate flat from whatever the endpoints happen to be.
        x = np.concatenate([[0.0], fp[r], [1.0]])
        y = np.concatenate([[0.0], tp[r], [1.0]])
        order = np.argsort(x, kind="stable")
        x, y = x[order], y[order]
        # np.interp needs strictly increasing x. Ties are one FPR reached at
        # several rho; keep the best TPR there, i.e. the upper envelope.
        x_u, first = np.unique(x, return_index=True)
        y_u = np.maximum.reduceat(y, first)
        tp_interp[r] = np.interp(fp_common, x_u, y_u)
    tp_se = (
        tp_interp.std(axis=0, ddof=1) / np.sqrt(n_rep) if n_rep > 1
        else np.zeros(n_points)
    )
    return fp_common, tp_interp.mean(axis=0), tp_se

In [34]:
filename = "fifty_simulations"
num_simulations=50
p=100
n=50
num_rho=30
# Bottom of every rho path: each grid keeps extending below its own
# min_ratio * rho_max until that replicate's FPR reaches FP_max, at most
# max_extra extra fits, so the curves all cover the same FPR range instead of
# stopping wherever the fixed grid ran out. FP_max = None restores the old
# fixed-length grids.
FP_max = 0.5
max_extra = 50

In [35]:
rng = np.random.default_rng()
mu_true  = rng.uniform(low=-5, high=5, size=p)
eta_true = rng.uniform(low=-5, high=5, size=p)
nu_true = rng.uniform(low=0.15, high=0.9, size=p)
Theta_true = dg.make_true_theta(p)

iu = np.triu_indices(p, k=1)
theoretical_rho = np.sqrt(np.log(p) / n)
true_pos_mask = Theta_true[iu] != 0
true_neg_mask = ~true_pos_mask

In [36]:
# Curves are ragged now: with FP_max set, each replicate stops at its own rho,
# so these are lists of 1D arrays rather than (num_simulations, num_rho) blocks.
# mean_roc and the [:k] slicing in save_curve take them as such.
fp_mwgp, tp_mwgp = [], []
fp_em_diag, tp_em_diag = [], []
fp_ggm, tp_ggm = [], []
fp_t, tp_t = [], []
fp_ts, tp_ts = [], []
auc_mwgp = np.zeros(num_simulations)
auc_em_diag = np.zeros(num_simulations)
auc_ggm = np.zeros(num_simulations)
auc_t = np.zeros(num_simulations)
auc_ts = np.zeros(num_simulations)
# One grid per method per replicate: rho_max is read off that method's own
# converged S_tau, so index i is the same relative position on the path
# (min_ratio ... 1 of rho_max) but not the same rho across methods.
rho_grids_mwgp = []
rho_grids_em_diag = []
rho_grids_ggm = []
rho_grids_t = []
rho_grids_ts = []

In [37]:
def save_curve(sim, filename=None):
    """Rebuild the figure from the first `sim` replicates already in memory.

    Nothing is refit: every stored curve list is sliced to [:sim]. mean_roc is
    a vertical average over replicates and each AUC was computed from its own
    replicate's curve, so a prefix of them is exactly the run you would have
    got had the loop stopped at `sim`. The fp/tp entries are ragged (FP_max
    stops each replicate at its own rho), which is why the slicing is all that
    happens here -- nothing stacks them.
    """
    k = int(sim)
    if not 1 <= k <= num_simulations:
        raise ValueError(f"sim must be in 1..{num_simulations}, got {sim}")

    def se(a):
        # ddof=1 is undefined for one replicate; report 0 rather than nan.
        return a.std(ddof=1) / np.sqrt(k) if k > 1 else 0.0

    fp_mwgp_k,     tp_mwgp_k     = fp_mwgp[:k],     tp_mwgp[:k]
    fp_em_diag_k,  tp_em_diag_k  = fp_em_diag[:k],  tp_em_diag[:k]
    fp_ggm_k,      tp_ggm_k      = fp_ggm[:k],      tp_ggm[:k]
    fp_t_k,        tp_t_k        = fp_t[:k],        tp_t[:k]
    fp_ts_k,       tp_ts_k       = fp_ts[:k],       tp_ts[:k]

    auc_mwgp_k    = auc_mwgp[:k]
    auc_em_diag_k = auc_em_diag[:k]
    auc_ggm_k     = auc_ggm[:k]
    auc_t_k       = auc_t[:k]
    auc_ts_k      = auc_ts[:k]

    grids_mwgp_k    = rho_grids_mwgp[:k]
    grids_em_diag_k = rho_grids_em_diag[:k]
    grids_ggm_k     = rho_grids_ggm[:k]
    grids_t_k       = rho_grids_t[:k]
    grids_ts_k      = rho_grids_ts[:k]

    fp_common, tp_mwgp_mean, tp_mwgp_se = mean_roc(fp_mwgp_k, tp_mwgp_k)
    _, tp_em_diag_mean, tp_em_diag_se = mean_roc(fp_em_diag_k, tp_em_diag_k)
    _, tp_ggm_mean, tp_ggm_se = mean_roc(fp_ggm_k, tp_ggm_k)
    _, tp_t_mean, tp_t_se = mean_roc(fp_t_k, tp_t_k)
    _, tp_ts_mean, tp_ts_se = mean_roc(fp_ts_k, tp_ts_k)

    print(f"\n(p={p}, n={n}) over {k} replicates")
    print(f"  Asymmetric model (MWGP): AUC = {auc_mwgp_k.mean():.3f} "
          f"(SE {se(auc_mwgp_k):.3f})")
    print(f"  Asymmetric model (Diagonal): AUC = {auc_em_diag_k.mean():.3f} "
          f"(SE {se(auc_em_diag_k):.3f})")
    print(f"  Classical t-model (TLASSO): AUC = {auc_t_k.mean():.3f} "
          f"(SE {se(auc_t_k):.3f})")
    print(f"  Alternative t-model (TSTAR_VARLASSO): AUC = {auc_ts_k.mean():.3f} "
          f"(SE {se(auc_ts_k):.3f})")
    print(f"  Naive Gaussian glasso (GGM): AUC = {auc_ggm_k.mean():.3f} "
          f"(SE {se(auc_ggm_k):.3f})")

    METHODS = (
        ("Asym. model MWGP", fp_mwgp_k, tp_mwgp_k, tp_mwgp_mean, tp_mwgp_se,
         auc_mwgp_k, grids_mwgp_k, COLOR_ASYM, "-"),
        ("Asym. diag. model", fp_em_diag_k, tp_em_diag_k, tp_em_diag_mean,
         tp_em_diag_se, auc_em_diag_k, grids_em_diag_k, COLOR_EM_DIAG, "-"),
        ("Naive GGM", fp_ggm_k, tp_ggm_k, tp_ggm_mean, tp_ggm_se, auc_ggm_k,
         grids_ggm_k, COLOR_GGM, "-."),
        ("Classical t-model", fp_t_k, tp_t_k, tp_t_mean, tp_t_se, auc_t_k,
         grids_t_k, COLOR_T, "-."),
        ("Alternative t-model", fp_ts_k, tp_ts_k, tp_ts_mean, tp_ts_se,
         auc_ts_k, grids_ts_k, COLOR_TS, "-."),
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color=COLOR_CHANCE)
    for name, fp_m, tp_m, tp_mean, tp_se, auc, grids, color, ls in METHODS:
        ax.plot(fp_common, tp_mean, color=color, linewidth=2, linestyle=ls,
                label=f"{name}, avg AUC={auc.mean():.3f}")
        ax.fill_between(fp_common, tp_mean - tp_se, tp_mean + tp_se,
                        color=color, alpha=0.15, linewidth=0)
        # The theoretical rho is a threshold, not a position on the averaged
        # curve: take it in each replicate and average those operating points.
        # It can sit slightly off the mean curve, which is honest -- that curve
        # is a vertical average and this point is not.
    ax.plot([], [], "o", color="#52514e",
            label=r"$\rho=\sqrt{\log p\,/\,n}$" + f" = {theoretical_rho:.3g}")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.2)
    ax.set_xlabel("false positive rate (1 - specificity)")
    ax.set_ylabel("true positive rate (sensitivity)")
    ax.set_title(f"Precision-matrix support recovery: p={p}, n={n}")
    ax.legend(loc="lower right")
    fig.tight_layout()

    out = (
        f'/content/drive/MyDrive/aat/results/roc/roc_{k}' if filename is None
        else f'/content/drive/MyDrive/aat/results/roc/{filename}_{k}'
    )
    fig.savefig(f"{out}.pdf")

In [38]:
# Output goes to roc_run.log so it does not flood the cell.
# Follow it live:  Get-Content roc_run.log -Wait -Tail 20
#
# Each method gets its own rho grid in each replicate: fit at the theoretical
# rho first, let it converge, then take rho_max = max_{i != j} |S_ij| off the
# converged S_tau. See pilot_rho_max for why a shared cov(Y) grid is wrong here.
#
# num_rho is only the starting length: each grid then extends itself downward
# until that method/replicate hits FPR >= FP_max (at most max_extra extra
# fits), so the stored curves and grids are ragged. See roc_curve_em.
kwargs_t = {"n_iter": 200, "verbose": False}
kwargs_ts = {"n_iter": 200, "verbose": False}
kwargs_em_diag = {"n_iter": 200, "verbose": False}
kwargs_mwgp = {
    "n_iter": 50,
    "verbose": False,
    "warning": True,
    "mcmc_samples": 150,
    "mcmc_thin": 1,
    "mcmc_warmup": 30,
    "proposal": "gig",
}

# with stream_to('/content/drive/MyDrive/aat/results/roc_run.log'):
for sim in range(num_simulations):
    print(f"Replicate {sim + 1}/{num_simulations}")

    # Generate data from an independent model (noisy skewed Gaussian)
    Y,_ = dg.simulate_aat_data(n, p, Theta_true, mu_true, eta_true, nu_true, rng)

    # Classical t-distribution model (run_tlasso)
    fp, tp, grid = roc_curve_em_autogrid(
        Y, tlasso.run_tlasso, true_pos_mask, true_neg_mask,
        pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=0.01,
        algorithm_kwargs=kwargs_t, fp_max=FP_max, max_extra=max_extra,
    )
    fp_t.append(fp); tp_t.append(tp); rho_grids_t.append(grid)
    auc_t[sim] = auc_from_curve(fp, tp)

    # Alternative t-distribution model (run_tstar_varlasso)
    fp, tp, grid = roc_curve_em_autogrid(
        Y, tlasso.run_tstar_varlasso, true_pos_mask, true_neg_mask,
        pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=0.01,
        algorithm_kwargs=kwargs_ts, fp_max=FP_max, max_extra=max_extra,
    )
    fp_ts.append(fp); tp_ts.append(tp); rho_grids_ts.append(grid)
    auc_ts[sim] = auc_from_curve(fp, tp)

    # Asymmetric Alternative t-distribution model (EM_DIAGONAL)
    fp, tp, grid = roc_curve_em_autogrid(
        Y, em.run_em_diagonal, true_pos_mask, true_neg_mask,
        pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=0.01,
        algorithm_kwargs=kwargs_em_diag, fp_max=FP_max, max_extra=max_extra,
    )
    fp_em_diag.append(fp); tp_em_diag.append(tp); rho_grids_em_diag.append(grid)
    auc_em_diag[sim] = auc_from_curve(fp, tp)

    # Asymmetric Alternative t-distribution model (EM_MWGP)
    fp, tp, grid = roc_curve_em_autogrid(
        Y, em.run_em_MWGP, true_pos_mask, true_neg_mask,
        pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=0.01,
        algorithm_kwargs=kwargs_mwgp, fp_max=FP_max, max_extra=max_extra,
    )
    fp_mwgp.append(fp); tp_mwgp.append(tp); rho_grids_mwgp.append(grid)
    auc_mwgp[sim] = auc_from_curve(fp, tp)

    # Naive Gaussian graphical lasso baseline
    fp, tp, grid = roc_curve_glasso_autogrid(
        Y, true_pos_mask, true_neg_mask, n_rho=num_rho, min_ratio=0.01,
        glasso_kwargs={"max_iter": 2000, "verbose": False},
        fp_max=FP_max, max_extra=max_extra,
    )
    fp_ggm.append(fp); tp_ggm.append(tp); rho_grids_ggm.append(grid)
    auc_ggm[sim] = auc_from_curve(fp, tp)

    save_curve(sim, filename = filename)

Replicate 1/50
  pilot fit of run_tlasso at rho=0.30349


/usr/local/lib/python3.12/dist-packages/sklearn/covariance/_graph_lasso.py:140: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.02763705127836147, tolerance: 0.016534139642016336
  coefs, _, _, _ = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.12/dist-packages/sklearn/covariance/_graph_lasso.py:140: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 11.53673023930969, tolerance: 0.02809363565110427
  coefs, _, _, _ = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.12/dist-packages/sklearn/covariance/_graph_lasso.py:140: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.5607596613117494, tolerance: 0.1597854764483772
  coefs, _, _, _ = cd_fast.enet_coordinate_descent_gram(
/usr/local/lib/python3.12/dist-packages/sklearn/covariance/_graph_lasso.py:140: ConvergenceWar

  -> rho_max = max|S_ij| (off-diag) = 296.89689
  rho grid: [296.89688686 253.30287935 216.10987359 184.37799674 157.30537951
 134.20789281 114.50185969  97.68930573  83.34537517  71.10759474
  60.6667139   51.75889001  44.15902103  37.67505714  32.14314759
  27.4235002   23.39684877  19.96143922  17.03045823  14.52983947
  12.39639194  10.57620309   9.02327648   7.69836941   6.5680013
   5.60360756   4.78081783   4.07884008   3.47993523   2.96896887]
Running run_tlasso over 30 rho values...
  rho=296.89689 (1/30)
0.00020580366330520683 0.0
  rho=253.30288 (2/30)
0.00041160732661041366 0.0
  rho=216.10987 (3/30)
0.0008232146532208273 0.0
  rho=184.37800 (4/30)
0.0010290183165260341 0.0
  rho=157.30538 (5/30)
0.001234821979831241 0.0
  rho=134.20789 (6/30)
0.0020580366330520683 0.0
  rho=114.50186 (7/30)
0.0030870549495781024 0.0
  rho=97.68931 (8/30)
0.004939287919324964 0.0
  rho=83.34538 (9/30)
0.0065857172257666186 0.0
  rho=71.10759 (10/30)
0.008643753858818687 0.0
  rho=60.66671 (

KeyboardInterrupt: 

In [ ]:
# Average the ROC curves across replicates and compute mean AUCs.
#
# Vertical averaging, not index-wise: the rho grids are replicate- and
# method-specific now, so a column of fp/tp is not a common threshold. See
# mean_roc. AUC is unaffected either way -- it is computed per replicate from
# that replicate's own curve and only then averaged.


def method_result(fp, tp, tp_mean, tp_se, auc, grids):
    """Per-method block for the JSON dump.

    The raw per-replicate fp/tp and rho grids go in alongside the averaged
    curve: the mean curve is derived now (a vertical average on `fp_common`),
    and nothing downstream could rebuild it -- or re-average it differently --
    from the summary alone.

    Everything per-replicate is dumped as a list of lists, not a rectangle:
    FP_max lets each replicate's grid stop at a different length.
    """
    grids = [np.asarray(g, dtype=float) for g in grids]
    # Geometric mean of the grids, over the prefix every replicate reached --
    # past that the average would be taken over a shrinking, self-selected
    # subset of replicates (the ones that needed the most extension).
    n_common = min(len(g) for g in grids)
    rho_grid_mean = np.exp(np.log(np.stack([g[:n_common] for g in grids])).mean(axis=0))
    return {
        "tp_mean": tp_mean.tolist(),
        "tp_se": tp_se.tolist(),
        "auc_mean": float(auc.mean()),
        "auc_se": float(auc.std(ddof=1) / np.sqrt(num_simulations)),
        "fp": [np.asarray(f, dtype=float).tolist() for f in fp],
        "tp": [np.asarray(t, dtype=float).tolist() for t in tp],
        "rho_grids": [g.tolist() for g in grids],
        "rho_grid_mean": rho_grid_mean.tolist()
    }


results = {
    "p": p,
    "n": n,
    "theoretical_rho": float(theoretical_rho),
    "fp_common": fp_common.tolist(),
    "asym_mwgp": method_result(
        fp_mwgp, tp_mwgp, tp_mwgp_mean, tp_mwgp_se, auc_mwgp, rho_grids_mwgp),
    "asym_em_diag": method_result(
        fp_em_diag, tp_em_diag, tp_em_diag_mean, tp_em_diag_se, auc_em_diag,
        rho_grids_em_diag),
    "ggm": method_result(
        fp_ggm, tp_ggm, tp_ggm_mean, tp_ggm_se, auc_ggm, rho_grids_ggm),
    "t": method_result(fp_t, tp_t, tp_t_mean, tp_t_se, auc_t, rho_grids_t),
    "ts": method_result(fp_ts, tp_ts, tp_ts_mean, tp_ts_se, auc_ts, rho_grids_ts),
}

with open(f"{filename}.json", "w") as f:
    json.dump(results, f)

NameError: name 'fp_mwgp' is not defined

In [ ]:
# Distribution of the simulated Y (the last replicate from the loop above).
# simulate_contaminated_normal_data replaces a fraction eps of individual
# ENTRIES with N(mu_star, contam_var) draws, so the contamination appears as a
# separate bump near mu_star rather than as a fattened tail.
from scipy.stats import norm, skew

sigma = np.linalg.inv(Theta_true)
mu_star = 2.5 * np.max(np.diag(sigma))          # mult * max(diag(sigma))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
BAR, REF, MARK = "#2a78d6", "#52514e", "#e34948"

ncol = 5
nrow = int(np.ceil(p / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 2.2 * nrow))
for j, ax in enumerate(axs.ravel()):
    if j >= p:
        ax.set_axis_off()
        continue
    yj = Y[:, j]
    ax.hist(yj, bins=60, density=True, color=BAR, edgecolor="white", linewidth=0.3)
    xs = np.linspace(yj.min(), yj.max(), 300)
    ax.plot(xs, norm.pdf(xs, yj.mean(), yj.std(ddof=1)),
            color=REF, linewidth=1.5, linestyle="--")
    ax.axvline(mu_star, color=MARK, linewidth=1.2)
    ax.set_title(f"$Y_{{{j + 1}}}$   skew {skew(yj):+.2f}", fontsize=9, color=INK)
    ax.tick_params(labelsize=7, colors=MUTED, length=3)
    ax.grid(axis="y", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)

# Legend at figure level: inside a panel it collides with the bars.
handles = [plt.Line2D([], [], color=REF, linestyle="--", linewidth=1.5),
           plt.Line2D([], [], color=MARK, linewidth=1.2)]
fig.legend(handles, ["Gaussian, matched mean/sd", r"$\mu_*$ (contamination centre)"],
           loc="lower center", ncol=2, frameon=False, fontsize=9, labelcolor=MUTED)

fig.suptitle(f"Simulated Y: n={n}, p={p}, contaminated normal "
             f"($\\mu_*$={mu_star:.2f})", fontsize=12, color=INK)
fig.tight_layout(rect=[0.02, 0.03, 1, 0.95])

# How much mass sits above mu_star beyond what a matched Gaussian predicts?
# (The uncontaminated body reaches past mu_star on its own, so the raw
# fraction above mu_star is NOT the contamination rate -- the excess is.)
obs = (Y > mu_star).mean()
exp = norm.sf(mu_star, Y.mean(), Y.std(ddof=1))
print(f"mu_star = {mu_star:.3f}")
print(f"  P(Y > mu_star): observed {obs:.3%}, matched Gaussian {exp:.3%}, "
      f"excess {obs - exp:+.3%}  (eps = 2%, half of it lands below mu_star)")
print(f"  pooled skew {skew(Y.ravel()):+.3f}   max |Y| {np.abs(Y).max():.2f}")


In [ ]:
# results = em.run_em_diagonal(Y, n_iter = 60, rho = rho_grid[5])